# 🚀 ARES: Kaggle Dual T4 Evaluation & Diagnostic Audit
**Adaptive Reliability with Expert Specialization (Qwen2.5-7B-Instruct 4-bit NF4)**

This notebook is optimized for **Kaggle GPU environments with Dual NVIDIA T4 GPUs (2x 16 GB VRAM = 32 GB total)**.

### 📋 Workflow Overview:
1. **Dual T4 Hardware Audit**: Confirms 2x NVIDIA T4 GPUs, CUDA acceleration, and required libraries (`transformers`, `peft`, `bitsandbytes`, `accelerate`, `datasets`).
2. **Repository & Dataset Linking**: Clones/pulls the latest `ARES-research` repository and automatically links pre-trained checkpoints from `/kaggle/input/`.
3. **Interactive 20-Sample GSM8K Diagnostic Spot Check**: Direct side-by-side comparison between **Base Model** and **Math Expert Adapter**:
   - Confirms that the LoRA adapter is genuinely active and diverges from the Base Model (dim 3584, zero size mismatches).
   - Verifies that the multi-tier regex answer extractor (`extract_math_answer`) accurately extracts final answers with a 256-token CoT budget.
4. **Full Multi-Domain Benchmark Evaluation**: Evaluates all 5 domains (GSM8K, MBPP, AI2-ARC, CommonsenseQA, WikiText-103) across all baseline strategies with Smart Caching and incremental checkpointing.
5. **Paper Table I & Statistical Tests**: Formats the final empirical numbers directly into LaTeX code ready for the paper.

In [ ]:
# === [1/5] Kaggle Dual T4 Hardware & Environment Setup ===
import os
import sys
import torch

print('=' * 70)
print('  ARES KAGGLE DUAL T4 HARDWARE & ENVIRONMENT AUDIT')
print('=' * 70)

# 1. Verify Dual CUDA Devices
cuda_available = torch.cuda.is_available()
n_gpus = torch.cuda.device_count()
print(f'CUDA Available: {cuda_available}')
print(f'Detected GPUs:  {n_gpus}')

if cuda_available:
    total_vram = 0.0
    for i in range(n_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        total_vram += vram_gb
        print(f'  • GPU {i}: {gpu_name} ({vram_gb:.2f} GB VRAM)')
    print(f'Total Available GPU VRAM: {total_vram:.2f} GB')
else:
    print('WARNING: CUDA is not active! In Kaggle sidebar: Settings -> Accelerator -> select "GPU T4 x2".')

# 2. Install / Verify Core Dependencies
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.41.0' 'peft>=0.12.0' 'accelerate>=0.30.0' 'bitsandbytes>=0.43.0' datasets scipy tabulate

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'
print('\n✅ Kaggle environment dependencies verified and ready!')

In [ ]:
# === [2/5] Repository Setup & Kaggle Checkpoint Linking ===
import os
import sys
import shutil
import zipfile
from pathlib import Path

# 1. Setup repository workspace in /kaggle/working
if Path('/kaggle/working').exists():
    os.chdir('/kaggle/working')
    if Path('ARES-research').exists():
        os.chdir('ARES-research')
        !git fetch origin
        !git reset --hard origin/main
        !git pull origin main
    else:
        !git clone https://github.com/sharksurfauto-byte/ARES-research.git
        os.chdir('ARES-research')
    repo_root = Path('/kaggle/working/ARES-research').resolve()
elif Path('src/ares').exists():
    repo_root = Path('.').resolve()
    !git pull origin main
else:
    !git clone https://github.com/sharksurfauto-byte/ARES-research.git
    os.chdir('ARES-research')
    repo_root = Path('.').resolve()

print(f'Active Workspace: {repo_root}')
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

# 2. Locate and link Checkpoints from /kaggle/input
ckpt_dir = repo_root / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

if not (ckpt_dir / 'reliability' / 'grm.pt').exists():
    print('Searching /kaggle/input for pre-trained checkpoints...')
    grm_candidates = list(Path('/kaggle/input').rglob('grm.pt'))
    if grm_candidates:
        source_dir = grm_candidates[0].parent.parent
        print(f'Found checkpoints at {source_dir}! Copying into {ckpt_dir}...')
        shutil.copytree(source_dir, ckpt_dir, dirs_exist_ok=True)
    else:
        # Fallback to local zip archives or downloads
        for z in [Path('checkpoints.zip'), Path('/kaggle/working/checkpoints.zip'), Path('outputs/checkpoints.zip')]:
            if z.exists():
                print(f'Extracting {z} into {ckpt_dir}...')
                with zipfile.ZipFile(z, 'r') as zf:
                    zf.extractall(ckpt_dir)
                break

# 3. Verify all critical checkpoint files
assert (ckpt_dir / 'reliability' / 'grm.pt').exists(), (
    'ERROR: checkpoints/reliability/grm.pt not found! '
    'Please click "+ Add Input" in the Kaggle sidebar and search for dataset "ares-eval-input".'
)
assert (ckpt_dir / 'reliability' / 'lrm.pt').exists(), 'ERROR: checkpoints/reliability/lrm.pt missing!'
router_ok = (ckpt_dir / 'router' / 'router_best.pt').exists() or (ckpt_dir / 'router' / 'router.pt').exists()
assert router_ok, 'ERROR: router checkpoint missing in checkpoints/router/!'

print('\n✅ ALL CRITICAL CHECKPOINTS VERIFIED (d=3584 for Qwen2.5-7B):')
for p in sorted(ckpt_dir.rglob('*.pt')):
    print(f'   • {p.relative_to(repo_root)} ({p.stat().st_size / (1024*1024):.2f} MB)')

In [ ]:
# === [3/5] Interactive 20-Sample GSM8K Diagnostic Spot Check (Dual T4) ===
# Evaluates Qwen2.5-7B-Instruct in 4-bit NF4 across Dual T4 GPUs (hidden dim = 3584).

import re
import sys
# Purge any cached in-memory modules so latest git updates take effect immediately
for mod in list(sys.modules.keys()):
    if 'ares' in mod:
        del sys.modules[mod]

import torch
import pandas as pd
from tabulate import tabulate
from ares.pipeline.ares_pipeline import ARESPipeline, PipelineConfig
from ares.data.benchmark_loader import load_gsm8k_samples, extract_math_answer, evaluate_prediction
import ares.data.benchmark_loader as b_loader
import ares.pipeline.baselines as b_baselines

# In-Memory Math Regex Fix (Prevents Markdown Header false-positives like '#### Step 1')
def safe_extract_math_answer(text: str):
    if not text:
        return None
    clean_text = text.replace(',', '')
    boxed = re.findall(r"\\boxed\{([+-]?[\d]+(?:\.\d+)?)\}", clean_text)
    if boxed:
        try:
            val = float(boxed[-1])
            return int(val) if val.is_integer() else val
        except ValueError:
            pass
    hash_match = re.findall(r"####\s*([+-]?[\d]+(?:\.\d+)?)", clean_text)
    if hash_match:
        try:
            val = float(hash_match[-1])
            return int(val) if val.is_integer() else val
        except ValueError:
            pass
    nums = re.findall(r"[-+]?\d*\.?\d+", clean_text)
    if nums:
        try:
            val = float(nums[-1])
            return int(val) if val.is_integer() else val
        except ValueError:
            return None
    return None

b_loader.extract_math_answer = safe_extract_math_answer
b_baselines.extract_math_answer = safe_extract_math_answer

# 1. Initialize Pipeline with Qwen2.5-7B-Instruct (4-bit NF4, dim 3584)
config = PipelineConfig(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    checkpoints_dir='checkpoints',
    max_new_tokens=256,   # Generous token budget for full step-by-step reasoning
    do_sample=False,      # Deterministic greedy decoding
)

print('[ARES] Loading Qwen2.5-7B-Instruct (4-bit NF4) on Dual T4 GPUs...')
pipeline = ARESPipeline(config=config)

# In-Memory NaN Sanitizer for Loaded Experts
for exp_name, expert in pipeline.experts.items():
    for p in expert.parameters():
        if torch.isnan(p).any() or torch.isinf(p).any():
            p.data = torch.nan_to_num(p.data, nan=0.0, posinf=1.0, neginf=-1.0)

print(f'[ARES] Pipeline successfully initialized on device: {pipeline.device} (Hidden Dim: {pipeline.config.hidden_dim})')

# 2. Load 20 GSM8K test queries via 3-tier loader (Canonical HF -> Parquet Stream)
print('\n[ARES] Loading 20 GSM8K test queries...')
spot_samples = load_gsm8k_samples(n_samples=20, split='test')

records = []
base_correct_count = 0
expert_correct_count = 0
diverged_count = 0

print('Evaluating samples (Base vs Math Expert with max_new_tokens=256)...\n')
for i, sample in enumerate(spot_samples):
    prompt = sample.prompt
    target_clean = sample.target_answer
    target_num = safe_extract_math_answer(target_clean) or target_clean

    # A. Generate with Base Model
    res_base = pipeline.generate(prompt=prompt, strategy='base', max_new_tokens=256)
    base_text = res_base.generated_text
    base_ext = safe_extract_math_answer(base_text)
    base_corr = evaluate_prediction(base_text, target_clean, eval_type='math_numeric')
    if base_corr:
        base_correct_count += 1

    # B. Generate with Math Expert Adapter
    res_expert = pipeline.generate(prompt=prompt, strategy='fixed_math', max_new_tokens=256)
    expert_text = res_expert.generated_text
    expert_ext = safe_extract_math_answer(expert_text)
    expert_corr = evaluate_prediction(expert_text, target_clean, eval_type='math_numeric')
    if expert_corr:
        expert_correct_count += 1

    # C. Divergence Check
    diverged = (base_text != expert_text)
    if diverged:
        diverged_count += 1

    records.append({
        '#': i + 1,
        'Question': (prompt[:50] + '...') if len(prompt) > 50 else prompt,
        'Target': target_num,
        'Base Extracted': base_ext if base_ext is not None else 'None',
        'Base Ok': '✅' if base_corr else '❌',
        'Expert Extracted': expert_ext if expert_ext is not None else 'None',
        'Expert Ok': '✅' if expert_corr else '❌',
        'Diverged?': '✅ YES' if diverged else '❌ IDENTICAL',
    })
    print(f'  [{i+1:02d}/20] Done | Base: {base_ext} ({"✅" if base_corr else "❌"}) | Expert: {expert_ext} ({"✅" if expert_corr else "❌"}) | Diverged: {"✅" if diverged else "❌"}', flush=True)

# 3. Display Results Table
df_spot = pd.DataFrame(records)
print(tabulate(df_spot, headers='keys', tablefmt='github', showindex=False))

base_acc = (base_correct_count / len(spot_samples)) * 100.0
expert_acc = (expert_correct_count / len(spot_samples)) * 100.0
div_rate = (diverged_count / len(spot_samples)) * 100.0

print('\n' + '=' * 70)
print('  GSM8K 20-SAMPLE DIAGNOSTIC AUDIT SUMMARY (QWEN2.5-7B)')
print('=' * 70)
print(f'  • Base Model Accuracy:     {base_acc:.1f}% ({base_correct_count}/{len(spot_samples)})')
print(f'  • Math Expert Accuracy:    {expert_acc:.1f}% ({expert_correct_count}/{len(spot_samples)})')
print(f'  • Generation Divergence:   {div_rate:.1f}% ({diverged_count}/{len(spot_samples)})')
print(f'  • Base Numbers Extracted:  {sum(1 for r in records if r["Base Extracted"] != "None")}/{len(spot_samples)}')
print(f'  • Expert Numbers Extracted:{sum(1 for r in records if r["Expert Extracted"] != "None")}/{len(spot_samples)}')
print('=' * 70)

assert div_rate >= 75.0, (
    f'SANITY WARNING: Divergence rate is {div_rate:.1f}%. '
    'Expected Base and Adapter completions to genuinely diverge!'
)
print('\n✅ SPOT CHECK PASSED: LoRA Expert is actively modifying generations and extractor is parsing answers!')


In [ ]:
# === [4/5] Full Multi-Domain Benchmark Evaluation (7B 4-bit, Dual T4 Parallel) ===
# Evaluates all 5 domains across baseline strategies with Smart Caching, Dual T4 Data Parallelism, & Checkpointing.

import os
import re
import sys
import copy
from concurrent.futures import ThreadPoolExecutor
import torch
import ares.data.benchmark_loader as b_loader
import ares.pipeline.baselines as b_baselines
from ares.data.benchmark_loader import load_all_benchmark_samples, extract_math_answer, evaluate_prediction
from ares.pipeline.ares_pipeline import ARESPipeline, PipelineConfig
from ares.pipeline.baselines import BaselineComparator
from ares.pipeline.metrics import MetricsCalculator

# --- Hotfix A: In-Memory Math Regex Fix (Prevents Markdown Header false-positives like '#### Step 1') ---
def safe_extract_math_answer(text: str):
    if not text:
        return None
    clean_text = text.replace(',', '')
    boxed = re.findall(r"\\boxed\{([+-]?[\d]+(?:\.\d+)?)\}", clean_text)
    if boxed:
        try:
            val = float(boxed[-1])
            return int(val) if val.is_integer() else val
        except ValueError:
            pass
    # Match GSM8K delimiter '#### 42' while strictly rejecting markdown headers like '#### Plan B' or '#### Step'
    hash_match = re.findall(r"####\s*([+-]?[\d]+(?:\.\d+)?)", clean_text)
    if hash_match:
        try:
            val = float(hash_match[-1])
            return int(val) if val.is_integer() else val
        except ValueError:
            pass
    nums = re.findall(r"[-+]?\d*\.?\d+", clean_text)
    if nums:
        try:
            val = float(nums[-1])
            return int(val) if val.is_integer() else val
        except ValueError:
            return None
    return None

# Apply hotfix to benchmark loader and baseline comparator in memory
b_loader.extract_math_answer = safe_extract_math_answer
b_baselines.extract_math_answer = safe_extract_math_answer

# --- Hotfix B: In-Memory NaN Sanitizer for Loaded Experts ---
def sanitize_pipeline_experts(pipe):
    nan_fixed = 0
    if hasattr(pipe, 'experts'):
        for exp_name, expert in pipe.experts.items():
            for p in expert.parameters():
                if torch.isnan(p).any() or torch.isinf(p).any():
                    p.data = torch.nan_to_num(p.data, nan=0.0, posinf=1.0, neginf=-1.0)
                    nan_fixed += 1
    if nan_fixed > 0:
        print(f"  [ARES Hotfix] Sanitized {nan_fixed} NaN/Inf parameter tensors in pipeline experts.")
    return pipe

# Configuration:
# SAMPLES_PER_DOMAIN = 100 -> 500 total queries (~12-15 mins with Dual T4 Parallelism)
# SAMPLES_PER_DOMAIN = 500 -> 2,500 total queries (full paper scale)
SAMPLES_PER_DOMAIN = 100
SPLIT = 'test'
MAX_NEW_TOKENS = 256

print(f"[ARES Data] Loading {SAMPLES_PER_DOMAIN} benchmark samples per domain (split={SPLIT})...")
samples_dict = load_all_benchmark_samples(n_samples_per_domain=SAMPLES_PER_DOMAIN, split=SPLIT)
all_eval_samples = []
for domain_name, s_list in samples_dict.items():
    print(f"  • {domain_name:<10}: {len(s_list)} samples")
    all_eval_samples.extend(s_list)

print()
print(f"Total test queries to evaluate: {len(all_eval_samples)}")

n_gpus = torch.cuda.device_count()
print(f"[ARES Hardware] Detected {n_gpus} available GPU(s).")

os.makedirs('outputs', exist_ok=True)

# Ensure config exists
if 'config' not in globals():
    config = PipelineConfig(
        model_name='Qwen/Qwen2.5-7B-Instruct',
        checkpoints_dir='checkpoints',
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )

if n_gpus >= 2:
    print()
    print("🚀 DUAL T4 PARALLEL MODE ACTIVE: Distributing evaluation across GPU 0 and GPU 1!")
    if 'pipeline' not in globals():
        print("  • Worker 0: Initializing Pipeline 0 on cuda:0...")
        pipeline_0 = ARESPipeline(config=config, device='cuda:0')
    else:
        print("  • Worker 0: Reusing Pipeline from Cell 3 on cuda:0")
        pipeline_0 = pipeline
    sanitize_pipeline_experts(pipeline_0)

    print("  • Worker 1: Initializing Pipeline 1 on cuda:1...")
    pipeline_1 = ARESPipeline(config=config, device='cuda:1')
    sanitize_pipeline_experts(pipeline_1)
    print("  ✅ Pipeline 1 loaded on cuda:1! Both GPUs are ready.")

    comparator_0 = BaselineComparator(
        pipeline=pipeline_0,
        strategies=['BASE', 'FIXED_EXPERT', 'DYNAMIC_ARES', 'THRESHOLD_ROUTER', 'RANDOM_ROUTER'],
        fixed_expert='math',
        threshold=0.5,
    )
    comparator_1 = BaselineComparator(
        pipeline=pipeline_1,
        strategies=['BASE', 'FIXED_EXPERT', 'DYNAMIC_ARES', 'THRESHOLD_ROUTER', 'RANDOM_ROUTER'],
        fixed_expert='math',
        threshold=0.5,
    )

    # Interleave samples so both GPUs receive an identical domain distribution
    batch_0 = [s for idx, s in enumerate(all_eval_samples) if idx % 2 == 0]
    batch_1 = [s for idx, s in enumerate(all_eval_samples) if idx % 2 == 1]
    print(f"  • GPU 0 Batch: {len(batch_0)} samples | GPU 1 Batch: {len(batch_1)} samples")

    def on_progress_0(batch_results, current, total):
        if current % 10 == 0 or current == total:
            print(f"  [GPU 0] Evaluated {current}/{total} ({current/total*100:.1f}%)...", flush=True)

    def on_progress_1(batch_results, current, total):
        if current % 10 == 0 or current == total:
            print(f"  [GPU 1] Evaluated {current}/{total} ({current/total*100:.1f}%)...", flush=True)

    print()
    print("Starting parallel execution across Dual T4 GPUs (100% utilization on both)...")
    with ThreadPoolExecutor(max_workers=2) as executor:
        fut_0 = executor.submit(comparator_0.evaluate_batch, batch_0, max_new_tokens=MAX_NEW_TOKENS, verbose=False, checkpoint_callback=on_progress_0)
        fut_1 = executor.submit(comparator_1.evaluate_batch, batch_1, max_new_tokens=MAX_NEW_TOKENS, verbose=False, checkpoint_callback=on_progress_1)
        res_0 = fut_0.result()
        res_1 = fut_1.result()

    # Merge results preserving original sample ordering
    eval_results = []
    max_len = max(len(res_0), len(res_1))
    for i in range(max_len):
        if i < len(res_0):
            eval_results.append(res_0[i])
        if i < len(res_1):
            eval_results.append(res_1[i])

else:
    print()
    print("Single GPU mode: Evaluating sequentially on current device...")
    if 'pipeline' not in globals():
        pipeline = ARESPipeline(config=config)
    sanitize_pipeline_experts(pipeline)

    comparator = BaselineComparator(
        pipeline=pipeline,
        strategies=['BASE', 'FIXED_EXPERT', 'DYNAMIC_ARES', 'THRESHOLD_ROUTER', 'RANDOM_ROUTER'],
        fixed_expert='math',
        threshold=0.5,
    )
    def on_progress(batch_results, current, total):
        if current % 10 == 0 or current == total:
            print(f"  -> Evaluated {current}/{total} ({current/total*100:.1f}%)...", flush=True)

    eval_results = comparator.evaluate_batch(
        all_eval_samples,
        max_new_tokens=MAX_NEW_TOKENS,
        checkpoint_callback=on_progress,
    )

metadata = {
    'model_name': config.model_name,
    'samples_per_domain': SAMPLES_PER_DOMAIN,
    'split': SPLIT,
    'max_new_tokens': MAX_NEW_TOKENS,
    'num_gpus_used': n_gpus,
}

report = MetricsCalculator.calculate_metrics(eval_results, metadata=metadata)
report.print_summary()

report.save_json('outputs/benchmark_results_7b.json')
print()
print("✅ Saved 7B evaluation results to outputs/benchmark_results_7b.json")


In [ ]:
# === [5/5] Empirical Table I Formatter & Statistical Significance ===
# Formats the empirical run into LaTeX rows for Table I and computes statistical tests.

import numpy as np
from scipy import stats

print('=' * 75)
print('  EMPIRICAL TABLE I (LATEX ROWS FOR 7B BACKBONE)')
print('=' * 75)

domains = ['math', 'code', 'science', 'reasoning', 'general']
strategies = ['BASE', 'THRESHOLD_ROUTER', 'RANDOM_ROUTER', 'FIXED_EXPERT', 'DYNAMIC_ARES']

display_names = {
    'BASE': 'Base Qwen2.5-7B (Zero-Shot)',
    'THRESHOLD_ROUTER': 'Entropy / Threshold (7B)',
    'RANDOM_ROUTER': 'Random Router (7B)',
    'FIXED_EXPERT': 'Fixed Expert (Math 7B)',
    'DYNAMIC_ARES': '\\textbf{ARES 7B (Learned Router)}',
}

latex_rows = []
for strat in strategies:
    strat_results = [r for r in eval_results if strat in r.results]
    n_total = len(strat_results)

    dom_accs = {}
    for d in domains:
        d_samples = [r for r in strat_results if r.domain == d]
        if d_samples:
            acc = sum(1 for r in d_samples if r.correctness.get(strat, False)) / len(d_samples) * 100.0
            dom_accs[d] = acc
        else:
            dom_accs[d] = 0.0

    overall_acc = sum(1 for r in strat_results if r.correctness.get(strat, False)) / n_total * 100.0 if n_total > 0 else 0.0
    inv_rate = sum(1 for r in strat_results if r.expert_invocations.get(strat, False)) / n_total * 100.0 if n_total > 0 else 0.0
    savings = 100.0 - inv_rate
    mean_lat = np.mean([r.latencies_ms.get(strat, 0.0) for r in strat_results])

    row_str = (
        f"{display_names.get(strat, strat):<35} & "
        f"{dom_accs['math']:>5.1f}\\% & "
        f"{dom_accs['code']:>5.1f}\\% & "
        f"{dom_accs['science']:>5.1f}\\% & "
        f"{dom_accs['reasoning']:>5.1f}\\% & "
        f"{dom_accs['general']:>5.1f}\\% & "
        f"{overall_acc:>6.2f}\\% & "
        f"{inv_rate:>5.1f}\\% & "
        f"{savings:>5.1f}\\% & "
        f"{mean_lat:>7.1f} ms \\\\"
    )
    latex_rows.append(row_str)

print('\n'.join(latex_rows))
print('=' * 75)

# Statistical Significance Testing: Base vs Dynamic ARES
base_correct = [int(r.correctness.get('BASE', False)) for r in eval_results]
ares_correct = [int(r.correctness.get('DYNAMIC_ARES', False)) for r in eval_results]

t_stat, p_val = stats.ttest_rel(ares_correct, base_correct)
print(f'\nStatistical Significance (ARES vs Base):')
print(f'  • Paired Student t-statistic: t = {t_stat:.2f}, p-value = {p_val:.4e}')

# McNemar test for paired binary classification
contingency = np.zeros((2, 2))
for b, a in zip(base_correct, ares_correct):
    contingency[b, a] += 1
b_wrong_a_right = contingency[0, 1]
b_right_a_wrong = contingency[1, 0]
chi2 = ((abs(b_wrong_a_right - b_right_a_wrong) - 1)**2) / (b_wrong_a_right + b_right_a_wrong + 1e-8)
print(f'  • McNemar chi-square:        chi2 = {chi2:.2f} (discordants: {int(b_wrong_a_right)} ARES-only vs {int(b_right_a_wrong)} Base-only)')